In [13]:
import os
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import DecisionTreeRegressor, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

if 'PYSPARK_SUBMIT_ARGS' in os.environ:
    del os.environ['PYSPARK_SUBMIT_ARGS']

spark = SparkSession.builder \
    .appName("ModeleImmoRhone") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [14]:
df = spark.read.csv(
    "logic_immo_lyon_propre.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

print(f"Lignes chargees : {df.count()}")
print(f"Colonnes        : {len(df.columns)}")
print()
df.printSchema()

UnsupportedOperationException: getSubject is not supported

In [15]:
print("=== Statistiques descriptives ===")
df.select("prix", "surface", "pieces", "chambres", "terrain").describe().show()

print("=== Repartition par type de bien ===")
df.groupBy("type_clean").count().orderBy(F.desc("count")).show()

print("=== Repartition par nombre de pieces ===")
df.groupBy("pieces").count().orderBy("pieces").show()

print("=== Top 10 villes ===")
df.groupBy("ville").count().orderBy(F.desc("count")).show(10)

=== Statistiques descriptives ===


NameError: name 'df' is not defined

In [ ]:
# Encodage des colonnes categoriques
indexer_ville = StringIndexer(inputCol="ville",       outputCol="ville_index",  handleInvalid="keep")
indexer_type  = StringIndexer(inputCol="type_clean",  outputCol="type_index",   handleInvalid="keep")
indexer_etage = StringIndexer(inputCol="etage_clean", outputCol="etage_index",  handleInvalid="keep")

# Liste des features
features_cols = [
    "surface", "pieces", "chambres", "terrain", "neuf_flag",
    "ville_index", "type_index", "etage_index"
]

assembler = VectorAssembler(inputCols=features_cols, outputCol="features")

print("Features utilisees :")
for f in features_cols:
    print(f"  - {f}")

In [ ]:
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

train_data = train_data.cache()
test_data  = test_data.cache()

print(f"Taille train : {train_data.count()} lignes")
print(f"Taille test  : {test_data.count()} lignes")

In [ ]:
dt = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="prix",
    maxDepth=8,
    seed=42
)

pipeline_dt = Pipeline(stages=[indexer_ville, indexer_type, indexer_etage, assembler, dt])

t0 = time.perf_counter()
model_dt = pipeline_dt.fit(train_data)
dt_train_time = time.perf_counter() - t0

print(f"Decision Tree - temps entrainement : {dt_train_time:.2f} s")

In [ ]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="prix",
    numTrees=100,
    maxDepth=10,
    seed=42
)

pipeline_rf = Pipeline(stages=[indexer_ville, indexer_type, indexer_etage, assembler, rf])

t0 = time.perf_counter()
model_rf = pipeline_rf.fit(train_data)
rf_train_time = time.perf_counter() - t0

print(f"Random Forest - temps entrainement : {rf_train_time:.2f} s")

In [ ]:
dt_pred = model_dt.transform(test_data)
rf_pred = model_rf.transform(test_data)

r2_eval   = RegressionEvaluator(labelCol="prix", predictionCol="prediction", metricName="r2")
rmse_eval = RegressionEvaluator(labelCol="prix", predictionCol="prediction", metricName="rmse")
mae_eval  = RegressionEvaluator(labelCol="prix", predictionCol="prediction", metricName="mae")

dt_r2   = r2_eval.evaluate(dt_pred)
dt_rmse = rmse_eval.evaluate(dt_pred)
dt_mae  = mae_eval.evaluate(dt_pred)

rf_r2   = r2_eval.evaluate(rf_pred)
rf_rmse = rmse_eval.evaluate(rf_pred)
rf_mae  = mae_eval.evaluate(rf_pred)

print("=" * 55)
print(f"{'Metrique':<20} {'Decision Tree':>15} {'Random Forest':>15}")
print("=" * 55)
print(f"{'R2':<20} {dt_r2:>15.4f} {rf_r2:>15.4f}")
print(f"{'RMSE (€)':<20} {dt_rmse:>15,.0f} {rf_rmse:>15,.0f}")
print(f"{'MAE (€)':<20} {dt_mae:>15,.0f} {rf_mae:>15,.0f}")
print(f"{'Temps train (s)':<20} {dt_train_time:>15.2f} {rf_train_time:>15.2f}")
print("=" * 55)

In [ ]:
print("A :")
rf_pred.select("prix", "prediction", "ville", "type_clean", "surface", "pieces") \
       .withColumn("ecart", F.round(F.col("prediction") - F.col("prix"), 0)) \
       .show(15)

In [ ]:
rf_model = model_rf.stages[-1]
importances = list(zip(features_cols, rf_model.featureImportances.toArray()))
importances.sort(key=lambda x: x[1], reverse=True)

print("Importance des variables (Random Forest) :")
print("-" * 40)
for feat, score in importances:
    bar = "█" * int(score * 50)
    print(f"  {feat:<20} : {score:.4f}  {bar}")